# Baseline GPT-2 cho Vietnamese Math Word Problems

Notebook này fine-tune `NlpHUST/gpt2-vietnamese` để sinh **cả lời giải toán tiếng Việt**, rồi kết thúc bằng câu đáp án cuối:

```text
Đáp án là: <số>
```

`model_output` khi dự đoán sẽ giống `sample_prediction.json`: một đoạn giải ngắn, không phải chỉ có mỗi câu đáp án.

Bản này dùng pipeline preprocessing trong `data preprocessing.md`: giữ `query_vi`, strip `[asy]`, chuẩn hóa decimal, rebuild target về anchor `Đáp án là:` và mask loss trên prompt. Theo yêu cầu hiện tại, notebook **bỏ qua Bước 4 — Deduplication và xử lý conflict**.


## Run

Kaggle: GPU ON, Internet OFF.

Notebook chỉ ghi các file cần thiết:

- Local: `data/train_preprocessed.json`; Kaggle: `/kaggle/working/data/train_preprocessed.json` — train set mới sau preprocessing và smart truncation token-aware.
- Local: `data/train_preprocessing_report.json`; Kaggle: `/kaggle/working/data/train_preprocessing_report.json` — thống kê preprocessing và một vài mẫu lỗi extract.
- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json`: output validation để tự kiểm tra.
- `test_predictions.json`: file nộp nếu có `test.json`.


In [1]:
# 1. Import và kiểm tra môi trường
import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import inspect
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

try:
    from IPython.display import display
except Exception:
    display = print

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
CUDA_OK = torch.cuda.is_available()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        capability = torch.cuda.get_device_capability(i)
        print(f"GPU {i}:", name, "| capability:", capability)
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        CUDA_OK = False
        print("WARNING: GPU hiện tại có compute capability < 7.0, không tương thích với PyTorch CUDA trong log Kaggle này.")
        print("Nếu muốn train trên Kaggle, hãy chọn GPU T4/V100/A100 thay vì P100. Notebook sẽ không train full được trên GPU này.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
GPU 0: Tesla T4 | capability: (7, 5)
GPU 1: Tesla T4 | capability: (7, 5)


In [2]:
# 2. Đường dẫn dữ liệu, model và output
# Nếu Kaggle mount input khác tên, chỉ cần sửa DATA_DIR hoặc MODEL_DIR ở đây.
def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))


def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    PROJECT_ROOT / "data",
)

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "baseline_gpt2_math"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle input là read-only, nên train mới sinh ra được ghi vào /kaggle/working/data.
GENERATED_DATA_DIR = WORK_DIR / "data" if IS_KAGGLE else PROJECT_ROOT / "data"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json")

OUTPUT_DIR = WORK_DIR / "gpt2_math_baseline_ckpt"
VALID_OUTPUT_PATH = WORK_DIR / "valid_output.json"
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"

SAFE_EOS_ID = 50256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)
print("GENERATED_DATA_DIR:", GENERATED_DATA_DIR)
print("TEST_FILE:", TEST_FILE)

DATA_DIR: /kaggle/input/datasets/kimanh2002/dataset-math
MODEL_DIR: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
WORK_DIR: /kaggle/working
GENERATED_DATA_DIR: /kaggle/working/data
TEST_FILE: None


In [3]:
# 3. Đọc dữ liệu
# Muốn chạy thử nhanh thì đổi các limit bên dưới thành số nhỏ, ví dụ 2000 và 100.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None


def load_records(path, need_response=False):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(line) for line in f if line.strip()]

    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu response_vi")
        item = dict(rec)
        item.setdefault("id", i)
        item.setdefault("type", "unknown")
        out.append(item)
    return out


raw_train = load_records(TRAIN_FILE, need_response=True)
raw_valid = load_records(VALID_FILE, need_response=True) if VALID_FILE.exists() else []
raw_test = load_records(TEST_FILE) if TEST_FILE else []

if MAX_TRAIN_SAMPLES is not None:
    raw_train = raw_train[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES is not None:
    raw_valid = raw_valid[:MAX_VALID_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test[:MAX_TEST_SAMPLES]

print("raw train:", len(raw_train))
print("raw valid:", len(raw_valid))
print("raw test :", len(raw_test))
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1400])

raw train: 100000
raw valid: 1000
raw test : 0
{
  "original_question_vi": "Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?",
  "original_question_en": "Bridgette and Alex are getting married. Bridgette is inviting 84 guests, and Alex is inviting two thirds of that number of guests. They hired a caterer to make a plated meal for each guest at the wedding reception. The caterer always makes ten extra plates just in case something goes wrong. Each plate of steak and asparagus in garlic butter will have 8 asparagus spears on it. How many asparagus spears will the caterer need in all?",
  "query_vi": "Bridgette và Alex sắp kết hôn. Bridget

In [4]:
# 4. Hàm trích đáp án và tính điểm
# Dùng cho cả data processing và validation.
ANSWER_ANCHORS = [
    r"Đáp án là\s*[:：]?",
    r"Câu trả lời là\s*[:：]?",
    r"(?:Câu\s+)?Trả lời(?:\s+là)?\s*[:：]?",
    r"Đáp án\s*[:：]?",
    r"The answer is\s*[:：]?",
    r"Answer\s*[:：]?",
    r"####\s*",
]
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，")
    return text or None


def extract_answer_text(text, allow_last_number=False):
    text = str(text or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        tail = text[matches[-1].end():]
        return clean_answer_tail(tail)

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    if allow_last_number:
        nums = NUM_RE.findall(text)
        if nums:
            return clean_answer_tail(nums[-1])
    return None

def parse_plain_number(text):
    text = str(text).strip().replace(" ", "")
    if not text:
        return None

    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a = parse_plain_number(parts[0])
            b = parse_plain_number(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None

    # Vietnamese/common formats:
    # 1.200 -> 1200, 1.200,5 -> 1200.5, 1,200 -> 1200, 1,200.5 -> 1200.5.
    if re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")

    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def parse_number(text):
    if text is None:
        return None

    text = str(text).strip()
    if not text:
        return None

    direct = parse_plain_number(text)
    if direct is not None:
        return direct

    m = NUM_RE.search(text)
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(rel_err, extractable=True):
    if not extractable or rel_err is None:
        return 0
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0


In [5]:
# 5. Data processing trước khi train
# Áp dụng Bước 1, 2, 3 trong data preprocessing.md.
# Bỏ qua Bước 4 theo yêu cầu: không dedup và không xử lý/drop conflict.
DROP_TRAIN_WITHOUT_FINAL_ANSWER = True
SAVE_PREPROCESSED_TRAIN = True
PREPROCESSED_TRAIN_FILE = GENERATED_DATA_DIR / "train_preprocessed.json"
PREPROCESSING_REPORT_FILE = GENERATED_DATA_DIR / "train_preprocessing_report.json"


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))


def normalized_hash(text):
    text = normalize_space(text).lower()
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).hexdigest()


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_records_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_jsonl_records(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def dataset_fingerprint(records):
    content = json.dumps(
        [r.get("query_vi", "") + "\n" + r.get("response_vi", "") for r in records],
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.md5(content).hexdigest()


def fix_artifacts(text):
    text = str(text or "")
    text = text.replace(r"\đóng hộp{", r"\boxed{")
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return text.strip()


def strip_asy_blocks(text):
    text = str(text or "")
    text = re.sub(r"\[asy\].*?\[/asy\]", "", text, flags=re.DOTALL | re.IGNORECASE)
    # Một số mẫu đã dịch hỏng nên còn "[asy]" nhưng mất "[/asy]". Cắt tới marker ngôn ngữ tự nhiên gần nhất.
    text = re.sub(
        r"\[asy\].*?(?=(?:Giá trị của|Giá trị là|Câu trả lời|Đáp án|Nếu chúng ta biết|Để giải|$))",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    text = re.sub(r"\[/asy\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def clean_text(text):
    text = str(text or "")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if re.search(r"Đáp án là|Câu trả lời là|####|\\boxed", text, flags=re.IGNORECASE):
        text = re.sub(
            r"\n(?:The answer is[:\s]+[\d.,/\\{}a-zA-Z]+\.?\s*)+$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    return text.strip()


def normalize_decimal_format(text):
    text = str(text or "")

    # European/Vietnamese mixed style: 1.200,5 -> 1200.5.
    text = re.sub(
        r"(?<![{\\])(\d+)\.(\d{3}),(\d{1,3})(?!\d)",
        lambda m: f"{m.group(1)}{m.group(2)}.{m.group(3)}",
        text,
    )

    # Decimal comma: 2,5 -> 2.5, -3,14 -> -3.14, 0,375 -> 0.375.
    # Tránh ngữ cảnh LaTeX rõ ràng như \frac{1,2}{3}.
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?0),(\d{1,3})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?\d+),(\d{1,2})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    return text

def preprocess_step2(query, response):
    query = strip_asy_blocks(query)
    response = strip_asy_blocks(response)
    query = clean_text(query)
    response = clean_text(response)
    query = normalize_decimal_format(query)
    response = normalize_decimal_format(response)
    return query, response


def extract_final_answer(response):
    text = str(response or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        return clean_answer_tail(text[matches[-1].end():])

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    numbers = re.findall(
        r"(?:\\frac\{[^}]+\}\{[^}]+\}|[-+]?\d+(?:[.,]\d+)?(?:\s*\\[a-zA-Z]+\{[^}]*\})*)",
        text,
    )
    if numbers:
        return clean_answer_tail(numbers[-1])
    return None

def normalize_answer(answer):
    answer = clean_answer_tail(answer) or ""
    answer = re.sub(r"\s+", " ", answer).strip()
    answer = re.sub(r"\(([-+]?\d+),([-+]?\d+)\)", r"(\1, \2)", answer)

    if not re.search(r"[\\{^_]", answer):
        # English thousands style in some MATH answers: 12,441,600 -> 12441600; 2,880 -> 2880.
        if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", answer) and not re.fullmatch(r"[-+]?0,\d{3}", answer):
            answer = answer.replace(",", "")
        # Vietnamese thousands style: 12.441.600 -> 12441600; 1.200,5 -> 1200.5.
        elif re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", answer):
            answer = answer.replace(".", "").replace(",", ".")
        else:
            answer = normalize_decimal_format(answer)
        answer = re.sub(r"^([-+]?\d[\d./]*)(?:\s+[a-zA-ZÀ-ỹ%].*)$", r"\1", answer)
    else:
        answer = normalize_decimal_format(answer)

    return answer.strip(" .。;；,，")

def rebuild_response(response, answer):
    cleaned = str(response or "").strip()
    cleaned = re.sub(
        r"\s*(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer)\s*[:：]?\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s*####\s*[^\n]*\s*$", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n?\s*\\boxed\s*\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}\s*[.。]?\s*$", "", cleaned)
    cleaned = cleaned.rstrip()
    return (cleaned + f"\nĐáp án là: {answer}").strip()


def preprocess_labeled_record(rec, idx, drop_without_answer):
    query_raw = fix_artifacts(rec.get("query_vi"))
    response_raw = fix_artifacts(rec.get("response_vi"))
    if not query_raw or not response_raw:
        return None, "missing_query_or_response"

    query, response = preprocess_step2(query_raw, response_raw)
    answer = normalize_answer(extract_final_answer(response))
    if not answer and drop_without_answer:
        return None, "extract_failed"

    if answer:
        response = rebuild_response(response, answer)

    item = {
        "id": rec.get("id", idx),
        "query_vi": query,
        "response_vi": response,
        "type": rec.get("type", "unknown"),
        "answer_text": answer or None,
        "answer_num": parse_number(answer) if answer else None,
    }
    return item, None


def process_train(records):
    kept = []
    drop_reasons = []
    failed_samples = []
    asy_stripped = 0

    for i, rec in enumerate(tqdm(records, desc="text preprocessing")):
        raw_joined = f"{rec.get('query_vi', '')}\n{rec.get('response_vi', '')}".lower()
        had_asy = "[asy]" in raw_joined
        item, reason = preprocess_labeled_record(rec, i, DROP_TRAIN_WITHOUT_FINAL_ANSWER)
        if reason:
            drop_reasons.append(reason)
            if reason == "extract_failed" and len(failed_samples) < 50:
                failed_samples.append({
                    "index": i,
                    "type": rec.get("type", "unknown"),
                    "query_vi": normalize_space(rec.get("query_vi"))[:180],
                    "response_tail": str(rec.get("response_vi", ""))[-300:],
                })
            continue
        if had_asy:
            asy_stripped += 1
        kept.append(item)

    return kept, Counter(drop_reasons), {
        "extract_failed_preview": failed_samples,
        "asy_stripped_count": asy_stripped,
    }


def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        query_raw = fix_artifacts(rec.get("query_vi"))
        query = normalize_decimal_format(clean_text(strip_asy_blocks(query_raw)))
        item = {
            "id": rec.get("id", i),
            "query_vi": query,
            "type": rec.get("type", "unknown"),
        }
        if has_response:
            response_raw = fix_artifacts(rec.get("response_vi"))
            _, response = preprocess_step2(query_raw, response_raw)
            answer = normalize_answer(extract_final_answer(response))
            if answer:
                response = rebuild_response(response, answer)
            item["response_vi"] = response
            item["answer_text"] = answer or None
            item["answer_num"] = parse_number(answer) if answer else None
        out.append(item)
    return out


train_records, drop_counter, preprocess_logs = process_train(raw_train)
valid_records = process_eval_or_test(raw_valid, has_response=True)
test_records = process_eval_or_test(raw_test, has_response=False)

preprocessing_report = {
    "train_before": len(raw_train),
    "train_after_text_preprocessing": len(train_records),
    "dropped_text_preprocessing": sum(drop_counter.values()),
    "drop_reasons_text_preprocessing": dict(drop_counter),
    "valid": len(valid_records),
    "test": len(test_records),
    "skip_deduplication_and_conflict_resolution": True,
    "fingerprint_text_preprocessing": dataset_fingerprint(train_records),
    **preprocess_logs,
}

print("train before:", len(raw_train), "| after text preprocessing:", len(train_records), "| dropped:", sum(drop_counter.values()))
print("drop reasons:", dict(drop_counter))
print("[asy] stripped in train:", preprocess_logs["asy_stripped_count"])
print("valid:", len(valid_records), "| test:", len(test_records))
print("File train mới sẽ được ghi sau smart truncation token-aware:", PREPROCESSED_TRAIN_FILE)
print("\nTarget sau xử lý:")
print(train_records[0]["response_vi"][:1000])


text preprocessing:   0%|          | 0/100000 [00:00<?, ?it/s]

train before: 100000 | after text preprocessing: 99993 | dropped: 7
drop reasons: {'extract_failed': 7}
[asy] stripped in train: 1012
valid: 1000 | test: 0
File train mới sẽ được ghi sau smart truncation token-aware: /kaggle/working/data/train_preprocessed.json

Target sau xử lý:
Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy tổng số khách là 84 + 56 = 140 khách. Người phục vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây.
Đáp án là: 1200


In [6]:
# 6. Kiểm tra dữ liệu sau processing
# Bước 4 dedup/conflict bị bỏ qua theo yêu cầu, nên duplicate/conflict chỉ được báo cáo để biết.
def feature_df(records, split):
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "split": split,
            "index": i,
            "type": rec.get("type", "unknown"),
            "query_words": word_count(rec.get("query_vi")),
            "response_words": word_count(rec.get("response_vi", "")),
            "has_answer": rec.get("answer_text") is not None,
            "answer_num": rec.get("answer_num"),
            "pair_hash": normalized_hash(rec.get("query_vi", "") + "\n" + rec.get("response_vi", "")),
            "ends_with_anchor": str(rec.get("response_vi", "")).rstrip().endswith("Đáp án là: " + str(rec.get("answer_text", ""))),
        })
    return pd.DataFrame(rows)


train_df = feature_df(train_records, "train")
valid_df = feature_df(valid_records, "valid") if valid_records else pd.DataFrame()

decimal_comma_pattern = re.compile(r"(?<!\d)\d+,\d{1,3}(?!\d)")
old_anchor_count = sum("Câu trả lời là" in r.get("response_vi", "") for r in train_records)
anchor_not_last = int((~train_df["ends_with_anchor"]).sum()) if len(train_df) else 0
asy_remaining = sum("[asy]" in (r.get("query_vi", "") + r.get("response_vi", "")).lower() for r in train_records)
bad_boxed_remaining = sum(r"\đóng hộp{" in r.get("response_vi", "") for r in train_records)
decimal_query_issues = sum(bool(decimal_comma_pattern.search(r.get("query_vi", ""))) for r in train_records)
decimal_response_issues = sum(bool(decimal_comma_pattern.search(r.get("response_vi", ""))) for r in train_records)
decimal_answer_issues = sum(bool(decimal_comma_pattern.search(str(r.get("answer_text", "")))) for r in train_records)

print("Phân bố type sau processing:")
display(train_df["type"].value_counts().rename_axis("type").reset_index(name="count"))

print("Độ dài train:")
display(train_df[["query_words", "response_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

print("Độ dài và answer rate theo type:")
by_type = (
    train_df.groupby("type")
    .agg(
        count=("index", "count"),
        query_p95=("query_words", lambda s: s.quantile(0.95)),
        response_p95=("response_words", lambda s: s.quantile(0.95)),
        answer_rate=("has_answer", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
display(by_type.round(3))

print("Tỷ lệ có final_answer:", round(float(train_df["has_answer"].mean()), 4))
print("Anchor không nằm cuối response:", anchor_not_last)
print("Còn anchor cũ 'Câu trả lời là':", old_anchor_count)
print("Còn [asy]:", asy_remaining)
print(r"Còn \đóng hộp{:", bad_boxed_remaining)
print("Decimal phẩy trong query:", decimal_query_issues)
print("Decimal phẩy trong response:", decimal_response_issues)
print("Decimal phẩy trong answer:", decimal_answer_issues)
print("Số cặp query-response trùng còn lại (không drop theo yêu cầu):", int(train_df["pair_hash"].duplicated().sum()))


Phân bố type sau processing:


,type,count
0,GSM_AnsAug,20329
1,GSM_Rephrased,20298
2,MATH_AnsAug,18987
3,MATH_Rephrased,12785
4,GSM_FOBAR,10191
5,GSM_SV,9920
6,MATH_FOBAR,3769
7,MATH_SV,3714


Độ dài train:


,query_words,response_words
count,99993.00,99993.00
mean,47.86,111.52
std,25.93,65.51
min,2.00,4.00
50%,45.00,95.00
90%,81.00,197.00
95%,94.00,234.00
99%,122.00,332.00
max,311.00,771.00


Độ dài và answer rate theo type:


,type,count,query_p95,response_p95,answer_rate
0,GSM_AnsAug,20329,91.00,151.0,1.0
2,GSM_Rephrased,20298,80.00,153.0,1.0
4,MATH_AnsAug,18987,62.00,170.0,1.0
6,MATH_Rephrased,12785,58.00,181.0,1.0
1,GSM_FOBAR,10191,120.00,234.5,1.0
3,GSM_SV,9920,103.00,278.0,1.0
5,MATH_FOBAR,3769,110.00,425.0,1.0
7,MATH_SV,3714,97.35,348.0,1.0


Tỷ lệ có final_answer: 1.0
Anchor không nằm cuối response: 0
Còn anchor cũ 'Câu trả lời là': 0
Còn [asy]: 0
Còn \đóng hộp{: 0
Decimal phẩy trong query: 179
Decimal phẩy trong response: 1143
Decimal phẩy trong answer: 0
Số cặp query-response trùng còn lại (không drop theo yêu cầu): 1189


In [7]:
# 7. Prompt, tokenizer, smart truncation và audit token length
MAX_LENGTH = 512
TOKEN_AUDIT_SAMPLES = 2000
PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải:\n"


def build_prompt(rec):
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())


tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer))
print("pad_token_id:", tokenizer.pad_token_id, "| eos_token_id:", tokenizer.eos_token_id)


def encode_no_special(text):
    return tokenizer(str(text or ""), add_special_tokens=False)["input_ids"]


def split_response_for_truncation(response, answer):
    response = str(response or "").rstrip()
    m = re.search(r"\nĐáp án là:\s*([^\n]+)\s*$", response, flags=re.IGNORECASE)
    if m:
        return response[:m.start()].rstrip(), "\nĐáp án là: " + m.group(1).strip()
    suffix = "\nĐáp án là: " + str(answer or extract_answer_text(response, allow_last_number=True) or "").strip()
    body = re.sub(r"\n?Đáp án là:\s*[^\n]+\s*$", "", response, flags=re.IGNORECASE).rstrip()
    return body, suffix


def measure_record_tokens(rec):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [SAFE_EOS_ID]
    return len(prompt_ids), len(response_ids), len(prompt_ids) + len(response_ids)


def smart_truncate_record(rec, max_length):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [SAFE_EOS_ID]
    original_length = len(prompt_ids) + len(response_ids)

    item = dict(rec)
    item["original_length"] = original_length
    item["was_truncated"] = False

    if original_length <= max_length:
        item["prompt_tokens"] = len(prompt_ids)
        item["response_tokens"] = len(response_ids)
        item["total_tokens"] = original_length
        return item, None

    body, suffix = split_response_for_truncation(rec.get("response_vi", ""), rec.get("answer_text"))
    suffix_ids = encode_no_special(suffix) + [SAFE_EOS_ID]
    middle_budget = max_length - len(prompt_ids) - len(suffix_ids)
    if middle_budget <= 0:
        return None, "too_long_prompt_or_answer"

    body_ids = encode_no_special(body)
    while True:
        kept_body_ids = body_ids[-middle_budget:] if len(body_ids) > middle_budget else body_ids
        body_text = tokenizer.decode(kept_body_ids).strip()
        new_response = (body_text.rstrip() + suffix) if body_text else suffix.lstrip()
        new_response_ids = encode_no_special(new_response) + [SAFE_EOS_ID]
        new_total = len(prompt_ids) + len(new_response_ids)
        if new_total <= max_length:
            item["response_vi"] = new_response
            item["was_truncated"] = True
            item["prompt_tokens"] = len(prompt_ids)
            item["response_tokens"] = len(new_response_ids)
            item["total_tokens"] = new_total
            return item, None
        overflow = new_total - max_length
        middle_budget -= max(1, overflow)
        if middle_budget <= 0:
            return None, "too_long_after_truncation"


def apply_token_length_policy(records, max_length):
    kept = []
    counter = Counter()
    examples = []
    for rec in tqdm(records, desc="smart truncation"):
        item, reason = smart_truncate_record(rec, max_length)
        if reason:
            counter[reason] += 1
            if len(examples) < 20:
                examples.append({
                    "id": rec.get("id"),
                    "type": rec.get("type"),
                    "reason": reason,
                    "query_vi": rec.get("query_vi", "")[:180],
                })
            continue
        if item.get("was_truncated"):
            counter["smart_truncated"] += 1
        kept.append(item)
    return kept, counter, examples


train_records, token_policy_counter, token_policy_examples = apply_token_length_policy(train_records, MAX_LENGTH)
preprocessing_report.update({
    "max_length": MAX_LENGTH,
    "train_after_token_policy": len(train_records),
    "dropped_token_policy": int(token_policy_counter.get("too_long_prompt_or_answer", 0) + token_policy_counter.get("too_long_after_truncation", 0)),
    "smart_truncated": int(token_policy_counter.get("smart_truncated", 0)),
    "token_policy_counter": dict(token_policy_counter),
    "token_policy_drop_preview": token_policy_examples,
    "fingerprint_final": dataset_fingerprint(train_records),
})

if SAVE_PREPROCESSED_TRAIN:
    save_records_jsonl(train_records, PREPROCESSED_TRAIN_FILE)
    save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
    print("Wrote:", PREPROCESSED_TRAIN_FILE)
    print("Wrote:", PREPROCESSING_REPORT_FILE)

    # Quan trọng: train trên đúng file train mới vừa sinh, không dùng raw train.json.
    train_records = load_jsonl_records(PREPROCESSED_TRAIN_FILE)
    TRAIN_SOURCE = f"preprocessed_file:{PREPROCESSED_TRAIN_FILE}"
else:
    TRAIN_SOURCE = "preprocessed_in_memory"

print("TRAIN_SOURCE:", TRAIN_SOURCE)
print("Train records used by Trainer:", len(train_records))
assert train_records, "Không có mẫu train sau preprocessing"
assert all("total_tokens" in r for r in train_records[: min(100, len(train_records))]), "Train records chưa qua token policy"
assert max(r.get("total_tokens", 0) for r in train_records) <= MAX_LENGTH, "Còn mẫu vượt MAX_LENGTH"

sample = train_records if len(train_records) <= TOKEN_AUDIT_SAMPLES else random.sample(train_records, TOKEN_AUDIT_SAMPLES)
token_rows = []
for rec in tqdm(sample, desc="token audit"):
    p_tokens, r_tokens, total_tokens = measure_record_tokens(rec)
    token_rows.append({
        "type": rec.get("type"),
        "prompt_tokens": p_tokens,
        "response_tokens": r_tokens,
        "total_tokens": total_tokens,
        "will_truncate": total_tokens > MAX_LENGTH,
        "was_truncated": bool(rec.get("was_truncated")),
    })

token_df = pd.DataFrame(token_rows)
display(token_df[["prompt_tokens", "response_tokens", "total_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
print("Tỷ lệ còn vượt MAX_LENGTH:", round(float(token_df["will_truncate"].mean()), 4))
print("Số mẫu smart truncated:", int(token_policy_counter.get("smart_truncated", 0)))
print("Số mẫu drop vì quá dài:", preprocessing_report["dropped_token_policy"])
display(token_df.groupby("type")[["will_truncate", "was_truncated"]].mean().sort_values("was_truncated", ascending=False))


Tokenizer vocab_size: 50257 | len: 50258
pad_token_id: 50256 | eos_token_id: 50256


smart truncation:   0%|          | 0/99993 [00:00<?, ?it/s]

Wrote: /kaggle/working/data/train_preprocessed.json
Wrote: /kaggle/working/data/train_preprocessing_report.json
TRAIN_SOURCE: preprocessed_file:/kaggle/working/data/train_preprocessed.json
Train records used by Trainer: 99993


token audit:   0%|          | 0/2000 [00:00<?, ?it/s]

,prompt_tokens,response_tokens,total_tokens
count,2000.00,2000.00,2000.00
mean,66.26,153.51,219.77
std,28.39,81.90,96.69
min,15.00,6.00,49.00
50%,63.00,133.00,197.00
90%,100.00,264.00,355.00
95%,118.00,321.05,418.05
99%,160.00,419.01,512.00
max,236.00,463.00,512.00


Tỷ lệ còn vượt MAX_LENGTH: 0.0
Số mẫu smart truncated: 2209
Số mẫu drop vì quá dài: 0


,will_truncate,was_truncated
type,,
MATH_FOBAR,0.0,0.173333
MATH_SV,0.0,0.114286
MATH_Rephrased,0.0,0.028269
GSM_SV,0.0,0.019139
MATH_AnsAug,0.0,0.005650
GSM_FOBAR,0.0,0.004630
GSM_Rephrased,0.0,0.000000
GSM_AnsAug,0.0,0.000000


In [8]:
# 8. Dataset cho supervised fine-tuning
# Loss chỉ tính trên phần lời giải, không tính trên prompt/padding.
def clamp_ids(ids, vocab_size):
    return [min(max(int(x), 0), vocab_size - 1) for x in ids]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids

    # Train records đã được smart truncate trước đó. Nhánh này là guard cho eval/debug:
    # giữ prompt nhiều nhất có thể và giữ đuôi response chứa dòng đáp án.
    room = max_length - len(prompt_ids)
    if room <= 0:
        prompt_ids = prompt_ids[: max_length - 1]
        room = max_length - len(prompt_ids)
    response_ids = response_ids[-room:] if room > 0 else []
    return prompt_ids, response_ids


class MathDataset(Dataset):
    def __init__(self, records, tokenizer, vocab_size, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.vocab_size = vocab_size
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
        response_ids = self.tokenizer(rec["response_vi"], add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)

        input_ids = clamp_ids(prompt_ids + response_ids, self.vocab_size)
        labels = [-100] * len(prompt_ids) + clamp_ids(response_ids, self.vocab_size)
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        max_len = int(math.ceil(max_len / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [ ]:
# 9. Tham số train và tạo Trainer
# Đây là các thông số thường đổi khi train.
RUN_TRAIN = True
EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 50
TRAINER_EVAL_SAMPLES = 1000

tmp_model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
tmp_model.config.pad_token_id = SAFE_EOS_ID
tmp_model.config.eos_token_id = SAFE_EOS_ID
MODEL_VOCAB_SIZE = tmp_model.get_input_embeddings().num_embeddings
del tmp_model
gc.collect()
torch.cuda.empty_cache()

train_ds = MathDataset(train_records, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)
eval_records_for_trainer = valid_records[:TRAINER_EVAL_SAMPLES]
eval_ds = MathDataset(eval_records_for_trainer, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH) if eval_records_for_trainer else None
collator = PadCollator()

effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count() if CUDA_OK else 0)
print("train source:", globals().get("TRAIN_SOURCE", "train_records"))
print("preprocessed train file:", PREPROCESSED_TRAIN_FILE)
print("train samples:", len(train_ds), "| eval samples:", len(eval_ds) if eval_ds else 0)
print("effective batch:", effective_batch)
print("steps/epoch:", math.ceil(len(train_ds) / effective_batch))


def make_training_args():
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=LOGGING_STEPS,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2 if IS_KAGGLE else 0,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch" if eval_ds else "no"
    else:
        kwargs["evaluation_strategy"] = "epoch" if eval_ds else "no"
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    if not CUDA_OK:
        if "use_cpu" in sig.parameters:
            kwargs["use_cpu"] = True
        elif "no_cuda" in sig.parameters:
            kwargs["no_cuda"] = True
    return TrainingArguments(**kwargs)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train source: preprocessed_file:/kaggle/working/data/train_preprocessed.json
preprocessed train file: /kaggle/working/data/train_preprocessed.json
train samples: 99993 | eval samples: 1000
effective batch: 64
steps/epoch: 1563


In [10]:
# 10. Train và lưu checkpoint
if RUN_TRAIN:
    if not CUDA_OK:
        raise RuntimeError(
            "Không có GPU CUDA dùng được cho full training. Nếu Kaggle đang cấp Tesla P100 như log, "
            "hãy đổi Accelerator sang T4/V100/A100. Nếu chỉ muốn chạy thử notebook, đặt RUN_TRAIN=False "
            "hoặc giảm MAX_TRAIN_SAMPLES xuống rất nhỏ."
        )
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        args=make_training_args(),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
    )

    start = time.time()
    train_output = trainer.train()
    print("Train minutes:", round((time.time() - start) / 60, 2))
    print(train_output)

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skip train. Inference sẽ dùng checkpoint nếu có, nếu không dùng base model.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.196750,1.197480


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train minutes: 132.28
TrainOutput(global_step=1563, training_loss=1.3344333742340635, metrics={'train_runtime': 7936.4209, 'train_samples_per_second': 12.599, 'train_steps_per_second': 0.197, 'total_flos': 1.9606896820224e+16, 'train_loss': 1.3344333742340635, 'epoch': 1.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
# 11. Hàm sinh lời giải
# Các thông số generation đặt ngay tại đây để dễ chỉnh khi test.
MAX_NEW_TOKENS = 192
NUM_BEAMS = 1
DO_SAMPLE = False
REPETITION_PENALTY = 1.05
NO_REPEAT_NGRAM_SIZE = 4


def save_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def postprocess_output(text):
    text = str(text).strip()
    for marker in ["\nCâu hỏi:", "\nQuestion:", "\n###"]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()
    return text


def generate_predictions(model_dir, records, output_path, name):
    device = "cuda" if CUDA_OK else "cpu"
    print("Load for generation:", model_dir, "| device:", device)

    gen_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
    gen_tokenizer.pad_token_id = SAFE_EOS_ID
    gen_tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(str(model_dir), torch_dtype=dtype, local_files_only=True).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    vocab_size = model.get_input_embeddings().num_embeddings
    outputs = []
    start_all = time.time()

    with torch.inference_mode():
        for i, rec in enumerate(tqdm(records, desc=name)):
            enc = gen_tokenizer(build_prompt(rec), return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
            input_ids = enc["input_ids"].clamp(max=vocab_size - 1)
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=enc.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=SAFE_EOS_ID,
                eos_token_id=SAFE_EOS_ID,
            )
            new_tokens = gen[0, input_ids.shape[1]:]
            text = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
            outputs.append({
                "id": rec.get("id", i),
                "query_vi": rec["query_vi"],
                "type": rec.get("type", "unknown"),
                "model_output": postprocess_output(text),
            })

    save_json(outputs, output_path)
    print("Wrote:", output_path)
    print("Minutes:", round((time.time() - start_all) / 60, 2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return outputs

In [12]:
# 12. Sinh output validation
RUN_VALIDATION = True
MODEL_FOR_INFERENCE = OUTPUT_DIR if OUTPUT_DIR.exists() else MODEL_DIR

if RUN_VALIDATION and valid_records:
    valid_outputs = generate_predictions(MODEL_FOR_INFERENCE, valid_records, VALID_OUTPUT_PATH, name="validation")
    print("\nOutput mẫu:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:1200])
else:
    valid_outputs = []
    print("Skip validation generation")

Load for generation: /kaggle/working/gpt2_math_baseline_ckpt | device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

validation:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote: /kaggle/working/valid_output.json
Minutes: 18.74

Output mẫu:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Nếu Susan đang chơi trò chơi cờ có 48 ô thì cô ấy cần tổng cộng 48 + 5 = 72 ô. Nếu cô ấy di dời hai ô thì cô cần 72 - 2 = 24 ô. Nếu Susan di chuyển hai lần thì cô cần 24 - 2 = 12 ô. Nếu có nhiều ô hơn cô ấy cần thì cô ấy sẽ cần 12 - 12 = 6 ô. Tổng số ô cô cần là 72 + 6 = 96 ô.\nĐáp án là: 96"
}


In [13]:
# 13. Đánh giá validation và in vài case để đọc lỗi
CASES_TO_SHOW = 8


def align_by_id(preds, golds):
    if all("id" in x for x in preds) and all("id" in x for x in golds):
        pred_map = {str(x["id"]): x for x in preds}
        return [(pred_map[str(g["id"])], g) for g in golds if str(g["id"]) in pred_map]
    return list(zip(preds, golds))


def evaluate_predictions(preds, golds):
    rows = []
    for row_index, (pred, gold) in enumerate(align_by_id(preds, golds)):
        gold_answer = extract_answer_text(gold.get("response_vi"), allow_last_number=True)
        pred_answer = extract_answer_text(pred.get("model_output"), allow_last_number=False)
        gold_num = parse_number(gold_answer)
        pred_num = parse_number(pred_answer)
        rel_err = relative_error(pred_num, gold_num)
        score = score_one(rel_err, pred_answer is not None)
        rows.append({
            "row_index": row_index,
            "id": gold.get("id"),
            "type": gold.get("type"),
            "query_vi": gold.get("query_vi"),
            "model_output": pred.get("model_output"),
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": rel_err,
            "extractable": pred_answer is not None,
            "score": score,
        })
    return rows


def score_summary(rows):
    n = len(rows)
    raw = sum(r["score"] for r in rows)
    return {
        "n": n,
        "raw_score": raw,
        "max_raw_score": 10 * n,
        "score_10": raw / n if n else 0,
        "extractable_rate": sum(r["extractable"] for r in rows) / n if n else 0,
        "buckets": {str(s): sum(r["score"] == s for r in rows) for s in [10, 5, 1, 0]},
    }


def show_cases(title, rows):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not rows:
        print("Không có case")
        return
    cols = ["row_index", "id", "type", "score", "rel_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
    display(pd.DataFrame(rows[:CASES_TO_SHOW])[cols])


if valid_outputs:
    eval_rows = evaluate_predictions(valid_outputs, valid_records)
    summary = score_summary(eval_rows)
    print("Validation summary:")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    by_type = pd.DataFrame(eval_rows).groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()
    display(by_type[["type", "n", "score_10", "extractable_rate", "raw_score", "max_raw_score", "buckets"]])

    show_cases("Một vài case đúng", [r for r in eval_rows if r["score"] == 10])
    show_cases("Một vài case sai", [r for r in eval_rows if r["score"] == 0])
    show_cases("Một vài case không tách được đáp án", [r for r in eval_rows if not r["extractable"]])
else:
    eval_rows = []
    summary = None
    print("Không có validation output để đánh giá")

Validation summary:
{
  "n": 1000,
  "raw_score": 514,
  "max_raw_score": 10000,
  "score_10": 0.514,
  "extractable_rate": 0.83,
  "buckets": {
    "10": 29,
    "5": 11,
    "1": 169,
    "0": 791
  }
}


/tmp/ipykernel_23/1188216732.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_type = pd.DataFrame(eval_rows).groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()


,type,n,score_10,extractable_rate,raw_score,max_raw_score,buckets
0,GSM_AnsAug,209,0.454545,0.956938,95,2090,"{'10': 3, '5': 5, '1': 40, '0': 161}"
1,GSM_FOBAR,122,0.475410,0.688525,58,1220,"{'10': 4, '5': 0, '1': 18, '0': 100}"
2,GSM_Rephrased,197,0.558376,0.974619,110,1970,"{'10': 5, '5': 3, '1': 45, '0': 144}"
3,GSM_SV,97,0.206186,0.360825,20,970,"{'10': 1, '5': 0, '1': 10, '0': 86}"
4,MATH_AnsAug,173,0.531792,0.872832,92,1730,"{'10': 5, '5': 3, '1': 27, '0': 138}"
5,MATH_FOBAR,45,0.844444,0.755556,38,450,"{'10': 3, '5': 0, '1': 8, '0': 34}"
6,MATH_Rephrased,116,0.577586,0.948276,67,1160,"{'10': 5, '5': 0, '1': 17, '0': 94}"
7,MATH_SV,41,0.829268,0.585366,34,410,"{'10': 3, '5': 0, '1': 4, '0': 34}"



Một vài case đúng


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,66,66,GSM_AnsAug,10,0.0,15,15,Matt có thể viết 10 từ một phút bằng tay phải và 7 từ một phút bằng tay trái. Matt sẽ viết bao nhiêu từ trong 5 phút bằng tay phải so với tay trái?,"Matt có thể viết được 10 từ một giây bằng tay phải. Anh ấy có thể viết một từ một phút, vì vậy anh ấy có thể làm việc 10 * 1 = 30 từ trong 5 giây. Anh ấy cũng có thể viết 7 từ ..."
1,171,171,MATH_FOBAR,10,0.0,1,1,"Giả sử $F_n$ là số Fibonacci thứ $n$, trong đó $F_1 = F_2 = 1$ và $F_{n + 1} = F_n + F_{n - 1}.$ Khi đó \[\prod_{k = 2}^{100} \left( \frac{F_k}{F_{k - 1}} - \frac{F_k}{F_{k + X...","Chúng ta được biết rằng $f_n = 1$ hoặc $f_2 = f_n + 1$. Thay thế các giá trị này vào phương trình, chúng ta nhận được $f_1 = f_2 + 1$. Chúng ta có thể viết lại phương trình dướ..."
2,173,173,GSM_Rephrased,10,0.0,4,4,"Nếu Vanessa đã sử dụng tổng cộng 800g đậu phộng đóng gói và cô ấy biết mình đã gửi 3 đơn hàng lớn, mỗi đơn hàng 200g, thì cô ấy đã gửi bao nhiêu đơn hàng nhỏ, mỗi đơn hàng 50g?","Nếu Vanessa đã gửi 3/3 = 2 đơn hàng lớn thì Vanessa đã gửi tổng cộng 2 * 2 = 4 đơn hàng nhỏ. Vì vậy, Vanessa đã gửi 4 đơn hàng lớn.\nĐáp án là: 4"
3,202,202,GSM_Rephrased,10,0.0,12,12,"Nếu Claire muốn làm 2 chiếc bánh cho mẹ cô ấy và mỗi chiếc bánh cần 2 gói bột mì, cô ấy sẽ chi bao nhiêu tiền cho bột mì, biết rằng mỗi gói có giá 3 đô la?","Nếu Claire muốn làm 1 chiếc bánh cho bà nội của mình thì cô ấy sẽ cần 2 * 2 = 4 chiếc bánh. Nếu Claire muốn mua 2 chiếc bánh, cô ấy cũng sẽ cần 4 * 3 = 6 chiếc bánh. Vì vậy, Cl..."
4,238,238,MATH_Rephrased,10,0.0,4\sqrt{5},4,Giá trị của |2 - 4i| + |2 + 4i|?,Chúng ta có thể viết lại phương trình dưới dạng $2 - 4 = \boxed{4}$.\nĐáp án là: 4
5,261,261,GSM_Rephrased,10,0.0,16,16,Nếu một hộp ban đầu chứa 2 tá bánh rán và gia đình đã ăn 8 chiếc bánh rán thì trong hộp còn lại bao nhiêu chiếc bánh rán?,"Nếu một hộp ban đẩu chứa 2 tá thì trong hộp có 2 tá nên số lượng bánh rán là 2 * 2 = 4 tá. Nếu gia đình ăn 8 chiếc thì trong hộp sẽ có 8 * 8 = 16 chiếc bánh rán. Vì vậy, trong ..."
6,271,271,MATH_AnsAug,10,0.0,0,0,"Hàm $f(x)$ lấy số thực dương thành số thực, sao cho \[xf(y) - yf(x) = f \left( \frac{x}{y} \right)\]với mọi số dương các số thực $x$ và $y.$ Tìm tất cả các giá trị có thể có củ...","Chúng ta có thể viết lại phương trình dưới dạng $f(f(x)) = f(x)$. Thay thế các giá trị đã cho, chúng ta nhận được $f(0) = f(0)$. Chia cả hai vế cho $0$, ta được $f(-0) = 0$. Do..."
7,282,282,GSM_FOBAR,10,0.0,2,2,Max thích sưu tập các mô hình xe lửa. Anh ấy yêu cầu một chiếc cho mỗi ngày sinh nhật của mình và yêu cầu x vào mỗi dịp Giáng sinh. Max luôn nhận được những món quà mà anh ấy y...,"Max đã mua một chiếc xe lửa vào cuối 5 năm. Anh ấy đã mua x chiếc xe lửa, vì vậy anh ấy đã mua 5 x x. Tổng số chuyến tàu là 5 * 5 = x. Chúng ta được biết rằng Max đã mua x, vì ..."



Một vài case sai


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,0,0,GSM_Rephrased,0,1.594595,37,96,"Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nh...",Nếu Susan đang chơi trò chơi cờ có 48 ô thì cô ấy cần tổng cộng 48 + 5 = 72 ô. Nếu cô ấy di dời hai ô thì cô cần 72 - 2 = 24 ô. Nếu Susan di chuyển hai lần thì cô cần 24 - 2 = ...
1,1,1,MATH_Rephrased,0,0.631579,19,7,"Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?","Chúng ta có thể viết lại phương trình dưới dạng $\angle PQr = \angle PAD = \angle PQR$. Đơn giản hóa, chúng ta nhận được $PAD = \boxed{7}$.\nĐáp án là: 7"
2,5,5,GSM_AnsAug,0,0.866667,90,12,"Hans đặt phòng ở khách sạn. Khách sạn có 10 tầng, mỗi tầng có 10 phòng giống nhau. Do xảy ra tai nạn nên tầng cuối cùng không còn chỗ cho khách. Xem xét không có khách nào khác...","Tầng 1 có 10 phòng, tầng 2 có 10 phòng và tầng 3 có 10 phòng. Tầng 4 có 10 phòng nên tầng 5 có 10 phòng còn lại. Tầng 6 có 10 phòng nhưng tầng 7 có 10 phòng thì tầng 8 có 10 ph..."
3,6,6,GSM_SV,0,NaN,12,None,Bà Dunbar đang cắm hoa cho đám cưới của cháu gái bà. Cô ấy cần làm 5 bó hoa và 7 món đồ trang trí bàn ăn. Cô sử dụng x bông hồng trắng để trang trí mỗi bàn và 5 bông hồng trắng...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số lượng bông hồng trắng mà cô ấy cần làm mỗi bàn. Hãy chia nhỏ thông tin đã cho: Số bông hồng trắng..."
4,7,7,GSM_FOBAR,0,NaN,9,None,"Grace bắt đầu công việc kinh doanh cảnh quan của riêng mình. Cô tính phí 6 đô la một giờ cho việc cắt cỏ, 11 đô la cho việc nhổ cỏ và x đô la cho việc phủ lớp phủ. Vào tháng 9,...","Grace bắt đầu công tác kinh doanh của mình vào tháng 9, vì vậy cô ấy đã chi 6 đô la cho công việc cắt cỏ. Cô ấy cũng tính phí cho việc nhổ lông, nhổ tóc và phủ lớp sơn. Tổng số..."
5,8,8,MATH_Rephrased,0,0.900000,60,6,"Xác định giá trị cao nhất trong số các bội số chung nhỏ nhất của 12 và 2, 12 và 4, 12 và 6, 12 và 8, 12 và 10, 12 và 12. Hãy thể hiện câu trả lời của bạn dưới dạng số nguyên.","Chúng ta có thể viết lại phương trình dưới dạng $12 + 2 = \boxed{8}$. Sử dụng công thức tính lũy thừa bậc hai, chúng ta có $12 + 8 = \box-6$. Đơn giản hóa, chúng ta nhận được $..."
6,9,9,GSM_AnsAug,0,0.600000,5,2,"Bob được hỗ trợ tiền thuê nhà vì anh ấy có thu nhập thấp. Nếu anh ta được tăng lương 0.50 USD/giờ và làm việc 40 giờ một tuần, anh ta sẽ thực sự kiếm được bao nhiêu tiền một tu...","Bob có thu nhập hàng tháng là 0.50 đô la/giờ, vì vậy anh ấy có thể làm việc 40/40 = 2 tuần một tuần. Anh ấy cũng có thu nhập từ việc làm thêm, vì vậy số tiền anh ấy kiếm được t..."
7,10,10,GSM_FOBAR,0,NaN,1,None,John phải thay vòng bi cho những chiếc máy mà anh ấy làm việc cùng. Anh ta có 10 chiếc máy và mỗi chiếc có 30 vòng bi. Thông thường nó có giá x $ cho mỗi ổ bi nhưng hiện tại đa...,"John có 10 chiếc vòng bi, mỗi chiếc có giá 0.75 đô la, vì vậy anh ấy có tổng cộng 10 * 0.75 = 1 đô la. Anh ấy cũng có 10 chiếc bánh xe, mỗi chiếc bánh có giá 0,75 đô la. Vì vậy..."



Một vài case không tách được đáp án


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,6,6,GSM_SV,0,None,12,None,Bà Dunbar đang cắm hoa cho đám cưới của cháu gái bà. Cô ấy cần làm 5 bó hoa và 7 món đồ trang trí bàn ăn. Cô sử dụng x bông hồng trắng để trang trí mỗi bàn và 5 bông hồng trắng...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số lượng bông hồng trắng mà cô ấy cần làm mỗi bàn. Hãy chia nhỏ thông tin đã cho: Số bông hồng trắng..."
1,7,7,GSM_FOBAR,0,None,9,None,"Grace bắt đầu công việc kinh doanh cảnh quan của riêng mình. Cô tính phí 6 đô la một giờ cho việc cắt cỏ, 11 đô la cho việc nhổ cỏ và x đô la cho việc phủ lớp phủ. Vào tháng 9,...","Grace bắt đầu công tác kinh doanh của mình vào tháng 9, vì vậy cô ấy đã chi 6 đô la cho công việc cắt cỏ. Cô ấy cũng tính phí cho việc nhổ lông, nhổ tóc và phủ lớp sơn. Tổng số..."
2,10,10,GSM_FOBAR,0,None,1,None,John phải thay vòng bi cho những chiếc máy mà anh ấy làm việc cùng. Anh ta có 10 chiếc máy và mỗi chiếc có 30 vòng bi. Thông thường nó có giá x $ cho mỗi ổ bi nhưng hiện tại đa...,"John có 10 chiếc vòng bi, mỗi chiếc có giá 0.75 đô la, vì vậy anh ấy có tổng cộng 10 * 0.75 = 1 đô la. Anh ấy cũng có 10 chiếc bánh xe, mỗi chiếc bánh có giá 0,75 đô la. Vì vậy..."
3,27,27,GSM_FOBAR,0,None,2,None,"Annie thích ăn bánh quy. Cô ấy ăn 5 cái bánh quy vào thứ Hai, gấp x lần vào thứ Ba và vào thứ Tư nhiều hơn 40% so với thứ Ba. Annie đã ăn bao nhiêu cái bánh quy trong ba ngày n...","Vào thứ Hai, Annie đã ăn 5 cái. Vào thứ Ba, cô ấy ăn gấp x lần, tức là 5 * x = 10 cái bánh quy. Vào thứ Tư, cô ấy đã ăn gấp x/2 lần, tức gấp 3 lần, gấp 4 lần, gấp 5 lần, gấp 6 ..."
4,29,29,GSM_FOBAR,0,None,10,None,"Henry muốn mua một chiếc áo phông, một chiếc quần jeans và một đôi tất. Chiếc quần jean có giá gấp đôi chiếc áo phông và chiếc áo phông đắt hơn đôi tất x $. Những chiếc tất có ...","Henry muốn mua một cái quần jean, một chiếc áo thun và một đôi giày. Anh ấy muốn mua một đôi tất, vì vậy anh ấy muốn mua đôi tất + 1 đôi tất + 2 đôi tất + 3 đôi tất + 4 đôi tất..."
5,36,36,MATH_Rephrased,0,None,16,None,"Chandra có bốn chiếc bát, mỗi chiếc có một màu khác nhau (đỏ, xanh dương, vàng và xanh lá cây), và cô ấy cũng có một chiếc ly phù hợp với màu của mỗi chiếc bát. Nếu cô ấy chọn ...","Chandra có 4 chiếc bát, trong đó có một màu đỏ và một màu xanh dương. Cô ấy có một chiếc bát và một chiếc cốc, vậy cô ấy có 4 chiếc * 2 chiếc bát = 4 chiếc bát. Cô ấy cũng có h..."
6,41,41,MATH_Rephrased,0,None,7,None,"Trong quỹ đạo hình elip của hành tinh Xavier, với mặt trời là một tiêu điểm, Xavier cách mặt trời 2 đơn vị thiên văn (AU) tại điểm gần nhất (cận điểm) và cách 12 AU tại điểm xa...","Mặt trời là một trong những đơn vị thiên nhiên lớn nhất trong vũ trụ. Vì vậy, nó có thể được tính bằng AU. Để tìm khoảng cách giữa mặt trời và mặt trời, chúng ta cần xác định k..."
7,46,46,GSM_FOBAR,0,None,50,None,Một quán cà phê có một số đơn hàng cần nấu và cũng cần làm bánh sandwich cho một nhóm khách hàng đến. 3 văn phòng địa phương mỗi văn phòng đã đặt mua 10 chiếc bánh mì kẹp thịt ...,"Quán cà phê có 3 văn phòng, mỗi văn phòng có 3 * (3/2) = 5 văn phòng. Mỗi văn phòng có 5 * (5/2) * (5 / 2) = 5 * (4/2) + 5 * (6/2) - 5 * (3 * (4 / 2) + 5/2) / 5 = 6 * (4 * (4 +..."


In [14]:
# 14. Sinh test_predictions.json cho Phase 2
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE and test_records:
    test_outputs = generate_predictions(MODEL_FOR_INFERENCE, test_records, TEST_OUTPUT_PATH, name="test")
    print("\nTest output mẫu:")
    print(json.dumps(test_outputs[:2], ensure_ascii=False, indent=2)[:1200])
else:
    print("Không có test.json, bỏ qua bước test inference")

Không có test.json, bỏ qua bước test inference


In [15]:
# 15. Kiểm tra file đầu ra chính
for p in [OUTPUT_DIR, VALID_OUTPUT_PATH, TEST_OUTPUT_PATH]:
    p = Path(p)
    if p.exists():
        size = p.stat().st_size if p.is_file() else "<dir>"
        print(p, "|", size)

print("\nDone.")

/kaggle/working/gpt2_math_baseline_ckpt | <dir>
/kaggle/working/valid_output.json | 819127

Done.
